# Generación de datos del TFM desde Google Drive

Este notebook monta Google Drive, localiza la carpeta `DATOS` y utiliza los módulos `.py` guardados allí:

- `catalogo_nodos.py`
- `descargar_coordenadas.py`
- `descargar_distancias.py`
- `descargar_pvgis.py`
- `generar_instancia.py`
- `generar_trazabilidad.py`
- `main_DATOS.py`

La estructura esperada en Drive es:

```text
Mi unidad/
└── TFM/
    └── DATOS/
        ├── catalogo_nodos.py
        ├── descargar_coordenadas.py
        ├── descargar_distancias.py
        ├── descargar_pvgis.py
        ├── generar_instancia.py
        ├── generar_trazabilidad.py
        ├── main_DATOS.py
        └── Figuras/
            └── Mapa_ProvinciasEspana.png
```

> Si tu carpeta está en otra ubicación, modifica únicamente `DATOS_DIR` en la celda de configuración.

In [1]:
# 1. Montar Google Drive
from google.colab import drive

drive.mount('/content/drive')

print('Google Drive montado correctamente.')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google Drive montado correctamente.


In [2]:
# 2. Configuración de rutas y carga de módulos
from pathlib import Path
import sys
import os
import importlib

# Cambia esta ruta si tu carpeta DATOS está en otra ubicación.
DATOS_DIR = Path('/content/drive/MyDrive/TFM/DATOS')

# Alternativa habitual si la carpeta está directamente dentro de Mi unidad:
# DATOS_DIR = Path('/content/drive/MyDrive/DATOS')

if not DATOS_DIR.exists():
    raise FileNotFoundError(
        f'No se encuentra la carpeta DATOS en: {DATOS_DIR}\n'
        'Modifica DATOS_DIR y vuelve a ejecutar esta celda.'
    )

# Añade DATOS al camino de importación para que main_DATOS.py
# pueda encontrar todos los módulos auxiliares.
if str(DATOS_DIR) not in sys.path:
    sys.path.insert(0, str(DATOS_DIR))

print(f'Carpeta de trabajo: {DATOS_DIR}')
print('Contenido encontrado:')
for elemento in sorted(DATOS_DIR.iterdir()):
    print('  -', elemento.name)

modulos_requeridos = [
    'catalogo_nodos.py',
    'descargar_coordenadas.py',
    'generar_instancia.py',
    'generar_trazabilidad.py',
    'main_DATOS.py',
]

faltan = [nombre for nombre in modulos_requeridos
          if not (DATOS_DIR / nombre).exists()]

if faltan:
    raise FileNotFoundError(
        'Faltan estos módulos en la carpeta DATOS: ' + ', '.join(faltan)
    )

# Importa el orquestador y sus módulos auxiliares desde Drive.
import main_DATOS
importlib.reload(main_DATOS)

print('Módulos cargados desde Google Drive correctamente.')

Carpeta de trabajo: /content/drive/MyDrive/TFM/DATOS
Contenido encontrado:
  - Figuras
  - Tablas
  - catalogo_nodos.py
  - descargar_coordenadas.py
  - descargar_distancias.py
  - descargar_pvgis.py
  - figuras
  - generar_instancia.py
  - generar_trazabilidad.py
  - main_DATOS.py
  - main_DATOS_colab.ipynb
  - tablas
Módulos cargados desde Google Drive correctamente.


In [4]:
# 4. Parámetros de ejecución

# True: regenera aunque ya existan los ficheros.
FORZAR = True

# False: usa Nominatim/OSRM/PVGIS cuando estén disponibles.
# True: usa cache/fallback, Haversine y perfil sintético sin consultas online.
OFFLINE = False

# Si no quieres utilizar una imagen de fondo concreta, deja None.
# La función buscará Figuras/Mapa_ProvinciasEspana.png.
MAPA_FONDO = None

# Extensión del mapa: [longitud mínima, longitud máxima,
#                     latitud mínima, latitud máxima]
EXTENT = [-9.55, 4.45, 35.85, 43.85]

print('FORZAR =', FORZAR)
print('OFFLINE =', OFFLINE)
print('MAPA_FONDO =', MAPA_FONDO)
print('EXTENT =', EXTENT)

FORZAR = True
OFFLINE = False
MAPA_FONDO = None
EXTENT = [-9.55, 4.45, 35.85, 43.85]


In [5]:
# 4. Ejecutar toda la generación de datos

# Se llama directamente a las funciones del main_DATOS.py, evitando argparse,
# que está pensado para ejecutar el archivo desde una terminal.
coords = main_DATOS.paso_coordenadas(
    offline=OFFLINE,
    forzar=FORZAR,
)

main_DATOS.paso_trazabilidad(
    offline=OFFLINE,
    forzar=FORZAR,
)

main_DATOS.paso_instancias(
    offline=OFFLINE,
    forzar=FORZAR,
)

main_DATOS.paso_mapa(
    coords=coords,
    mapa_fondo=MAPA_FONDO,
    extent=EXTENT,
    forzar=FORZAR,
)

main_DATOS.paso_perfil(forzar=FORZAR)

print('\n' + '=' * 60)
print('GENERACIÓN COMPLETADA')
print('=' * 60)
print('Tablas:', main_DATOS.CARPETA_TABLAS)
print('Figuras:', main_DATOS.CARPETA_FIGURAS)

[>] Geocodificando coordenadas de los nodos...
    45 nodos disponibles.
[>] Generando trazabilidad de coordenadas...
Trazabilidad generada -> /content/drive/MyDrive/TFM/DATOS/Tablas/trazabilidad_coordenadas.csv
  15 plantas + 30 clientes = 45 nodos
[>] Generando instancia_small (dist=osrm, ren=pvgis)...
    -> /content/drive/MyDrive/TFM/DATOS/Tablas/instancia_small.json  (HTotal=22400 kg/dia)
[>] Generando instancia_medium (dist=osrm, ren=pvgis)...
    -> /content/drive/MyDrive/TFM/DATOS/Tablas/instancia_medium.json  (HTotal=38600 kg/dia)
[>] Generando instancia_large (dist=osrm, ren=pvgis)...
    -> /content/drive/MyDrive/TFM/DATOS/Tablas/instancia_large.json  (HTotal=53000 kg/dia)
[>] Dibujando mapa de nodos geocodificados...
    fondo: /content/drive/MyDrive/TFM/DATOS/Figuras/Mapa_ProvinciasEspana.png  extent=[-9.55, 4.45, 35.85, 43.85]
    [aviso] adjustText no instalado; aplico un offset simple.
    -> /content/drive/MyDrive/TFM/DATOS/Figuras/HV_D_MapaNodos.png
[>] Dibujando perf

In [6]:
# 5. Comprobar y listar los resultados generados

from pathlib import Path

carpetas_salida = [
    Path(main_DATOS.CARPETA_TABLAS),
    Path(main_DATOS.CARPETA_FIGURAS),
]

for carpeta in carpetas_salida:
    print(f'\n{carpeta}:')
    if carpeta.exists():
        for fichero in sorted(carpeta.iterdir()):
            if fichero.is_file():
                print(f'  - {fichero.name} ({fichero.stat().st_size / 1024:.1f} KB)')
    else:
        print('  [aviso] La carpeta todavía no existe.')


/content/drive/MyDrive/TFM/DATOS/Tablas:
  - coords_cache.json (2.1 KB)
  - instancia_large.json (71.2 KB)
  - instancia_medium.json (32.1 KB)
  - instancia_small.json (10.4 KB)
  - trazabilidad_coordenadas.csv (5.9 KB)

/content/drive/MyDrive/TFM/DATOS/Figuras:
  - HV_D_MapaNodos.png (1151.6 KB)
  - HV_D_PerfilRenovable.png (160.1 KB)
  - Mapa_ProvinciasEspana.png (465.2 KB)


## Ejecuciones posteriores

- Para reutilizar los resultados existentes, establece `FORZAR = False`.
- Para regenerar todo, establece `FORZAR = True`.
- Para trabajar sin internet, establece `OFFLINE = True`.
- Las salidas se guardan directamente en las carpetas `Tablas/` y `Figuras/` dentro de `DATOS` en Google Drive.

Si aparece `ModuleNotFoundError`, comprueba que el nombre del archivo `.py` coincide exactamente con el que importa `main_DATOS.py` y que todos los módulos están dentro de la misma carpeta `DATOS`.